In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt


In [2]:
######PART : CALCULATE THE TOTAL NUMBER OF STUDENTS BY DEMOGRAPHIC GROUP AND YEAR FOR EVERY HIGH SCHOOL######

demo = pd.read_csv("/Users/abrahamsadat/My Drive/CoffeeCoders/data/panel_yx_highschools_base_new.csv")

#Store all the demographics by percent in a list called "pct_vars"
pct_vars = [
    'x_pct_asian',
    'x_pct_black',
    'x_pct_el',
    'x_pct_hispanic',
    'x_pct_homeless',
    'x_pct_iep',
    'x_pct_low_income',
    'x_pct_white'
]

#Extract the school identifier and year columns and put them in a list called "id_cols"
id_cols = ['school_id', 'school_name', 'year']  # <-- change 'school_id' to your actual school identifier

#Keep only what we need
df = demo[id_cols + ['x_enrollment'] + pct_vars].dropna(subset=['x_enrollment'])

#Convert % -> proportions
props = df[pct_vars].div(100)

#Turn each % into a count = proportion * enrollment
counts = props.mul(df['x_enrollment'], axis=0)

#Give the count columns nicer names
counts.columns = [c.replace('x_pct_', 'n_') for c in counts.columns]

school_year_counts = pd.concat(
    [df[id_cols + ['x_enrollment']], counts],
    axis=1
)

#Round counts to whole students
school_year_counts[counts.columns] = school_year_counts[counts.columns].round().astype('Int64')

#Take a glance
school_year_counts.head()

,school_id,school_name,year,x_enrollment,n_asian,n_black,n_el,n_hispanic,n_homeless,n_iep,n_low_income,n_white
0,05-016-2020-17-0001,Evanston Twp High School,2016,3322.0,166,1010,120,581,166,442,1325,1438
1,05-016-2020-17-0001,Evanston Twp High School,2017,3285.0,184,966,138,591,164,397,1330,1449
2,05-016-2020-17-0001,Evanston Twp High School,2018,3459.0,197,955,159,636,166,401,1328,1570
3,05-016-2020-17-0001,Evanston Twp High School,2019,3514.0,197,959,183,647,144,411,1283,1606
4,05-016-2020-17-0001,Evanston Twp High School,2020,3597.0,209,935,183,673,112,399,1248,1647


In [3]:
pct_vars = [
    'x_pct_asian',
    'x_pct_black',
    'x_pct_el',
    'x_pct_hispanic',
    'x_pct_homeless',
    'x_pct_iep',
    'x_pct_low_income',
    'x_pct_white'
]

def compute_county_counts(group):
    group = group[pct_vars + ['x_enrollment']].dropna()

    props = group[pct_vars] / 100
    counts = props.multiply(group['x_enrollment'], axis=0)

    county_counts = counts.sum()
    county_counts.index = county_counts.index.str.replace('x_pct_', 'n_', regex=False)  # rename labels

    return county_counts

county_year_counts = (
    demo.groupby('year')
        .apply(compute_county_counts)
        .reset_index()
)

county_year_counts = (
    school_year_counts
    .groupby("year", as_index=False)
    .agg(
        n_black=("n_black", "sum"),
        n_white=("n_white", "sum"),
        n_hispanic=("n_hispanic", "sum"),
        n_low_income=("n_low_income", "sum"),
        x_enrollment=("x_enrollment", "sum")
    )
)

#Create not-low-income counts
county_year_counts["n_not_low_income"] = (
    county_year_counts["x_enrollment"] - county_year_counts["n_low_income"]
).clip(lower=0)

/var/folders/5k/2ty13j0s5y70zwdnmbbzwh_r0000gn/T/ipykernel_32656/3514310485.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_county_counts)


In [ ]:
######BUILD A CLEAN DISSIMILARITY INDEX PIPELINE — SCHOOL-YEAR CONTRIBUTIONS + COUNTY-YEAR D #####

###### PART 1: FUNCTION THAT COMPUTES A TWO-GROUP DISSIMILARITY INDEX (D) FOR ONE YEAR AT A TIME ######

#This computes per the school’s contribution term (how much it contributes to segregation in the county overall):
#contrib_i = 0.5 * | (a_i/A) - (b_i/B) |
#Then D_year = sum_i contrib_i

def build_dissimilarity_school_year(
    school_year_counts: pd.DataFrame,
    county_year_counts: pd.DataFrame,
    group_a_col: str,
    group_b_col: str,
    *,
    school_id_col: str = "school_id",
    school_name_col: str = "school_name",
    year_col: str = "year",
    enrollment_col: str = "x_enrollment",
    pair_name: str = "A_vs_B",
    fill_missing_with_zero: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    #Keep only what we need 
    needed_school_cols = [school_id_col, school_name_col, year_col, enrollment_col, group_a_col, group_b_col]
    school_year_pair = school_year_counts[needed_school_cols].copy()

    #Treat missing schools and demographic group counts as 0s:
    if fill_missing_with_zero:
        school_year_pair[[group_a_col, group_b_col]] = school_year_pair[[group_a_col, group_b_col]].fillna(0)

    #Pull the county totals for the same two groups and rename them clearly
    county_totals_pair = (
        county_year_counts[[year_col, group_a_col, group_b_col]]
        .rename(columns={group_a_col: "A_total", group_b_col: "B_total"})
        .copy()
    )

    #Attach county totals to each school-year row (so every school-year can compute a_i/A and b_i/B)
    school_year_with_totals = school_year_pair.merge(county_totals_pair, on=year_col, how="left")

    #Drop cases where the index is undefined (missing totals or zero totals)
    school_year_with_totals = school_year_with_totals.dropna(subset=["A_total", "B_total"]).copy()
    school_year_with_totals = school_year_with_totals[
        (school_year_with_totals["A_total"] > 0) & (school_year_with_totals["B_total"] > 0)
    ].copy()

    #Compute each school’s share of the countywide population for each group
    school_year_with_totals["share_A_in_school"] = school_year_with_totals[group_a_col] / school_year_with_totals["A_total"]
    school_year_with_totals["share_B_in_school"] = school_year_with_totals[group_b_col] / school_year_with_totals["B_total"]

    #Compute the contribution term for this school-year
    school_year_with_totals["seg_contrib"] = 0.5 * (
        school_year_with_totals["share_A_in_school"] - school_year_with_totals["share_B_in_school"]
    ).abs()

    #Sum contributions across schools to get the county-year dissimilarity index (D)
    D_by_year = (
        school_year_with_totals.groupby(year_col, as_index=False)["seg_contrib"]
        .sum()
        .rename(columns={"seg_contrib": "D"})
    )

    #Attach D back to each school-year row (useful for plotting / QA)
    school_year_with_totals = school_year_with_totals.merge(D_by_year, on=year_col, how="left")

    #Add a couple labels so stacked results stay readable
    school_year_with_totals["pair"] = pair_name
    school_year_with_totals["group_a"] = group_a_col
    school_year_with_totals["group_b"] = group_b_col

    return school_year_with_totals, D_by_year


######PART 2: CREATE "NOT LOW-INCOME" COUNTS (LOW-INCOME VS NOT IS A BINARY SPLIT) ######

school_year_counts_aug = school_year_counts.copy()
county_year_counts_aug = county_year_counts.copy()

#Treat missing values as zero before constructing complements
school_year_counts_aug["n_low_income"] = school_year_counts_aug["n_low_income"].fillna(0)
county_year_counts_aug["n_low_income"] = county_year_counts_aug["n_low_income"].fillna(0)

#School-level "not low-income" is the remainder of enrollment
school_year_counts_aug["n_not_low_income"] = (
    school_year_counts_aug["x_enrollment"] - school_year_counts_aug["n_low_income"]
).clip(lower=0)

#County-level total enrollment by year (needed to build county "not low-income")
county_total_enrollment = (
    school_year_counts_aug.groupby("year", as_index=False)["x_enrollment"].sum()
    .rename(columns={"x_enrollment": "county_total_enrollment"})
)

#Attach county enrollment totals and build county "not low-income"
county_year_counts_aug = county_year_counts_aug.merge(county_total_enrollment, on="year", how="left")
county_year_counts_aug["n_not_low_income"] = (
    county_year_counts_aug["county_total_enrollment"] - county_year_counts_aug["n_low_income"]
).clip(lower=0)


######PART 3: RUN THREE TWO-GROUP DISSIMILARITY INDICES ######
#1) Black vs White
#2) White vs Hispanic
#3) Low-income vs Not low-income

bw_school_year, bw_D_by_year = build_dissimilarity_school_year(
    school_year_counts=school_year_counts_aug,
    county_year_counts=county_year_counts_aug,
    group_a_col="n_black",
    group_b_col="n_white",
    pair_name="black_white",
)

wh_school_year, wh_D_by_year = build_dissimilarity_school_year(
    school_year_counts=school_year_counts_aug,
    county_year_counts=county_year_counts_aug,
    group_a_col="n_white",
    group_b_col="n_hispanic",
    pair_name="white_hispanic",
)

li_school_year, li_D_by_year = build_dissimilarity_school_year(
    school_year_counts=school_year_counts_aug,
    county_year_counts=county_year_counts_aug,
    group_a_col="n_low_income",
    group_b_col="n_not_low_income",
    pair_name="lowincome_notlowincome",
)


######PART 4: STACK SCHOOL-YEAR RESULTS ######
#Each row is one school-year for one pair, with:
#seg_contrib (school contribution) and D (county-year index)

school_year_segregation_all = pd.concat(
    [bw_school_year, wh_school_year, li_school_year],
    ignore_index=True
)

#Keep the columns we actually use downstream (cleaner tables, easier debugging)
school_year_segregation_all_long = school_year_segregation_all[
    ["school_id", "school_name", "year", "x_enrollment",
     "group_a", "group_b", "A_total", "B_total",
     "share_A_in_school", "share_B_in_school",
     "seg_contrib", "D", "pair"]
].copy()


######PART 5: BUILD A WIDE COUNTY-YEAR TABLE (ONE ROW PER YEAR) ######

bw_D_by_year = bw_D_by_year.rename(columns={"D": "D_black_white"})
wh_D_by_year = wh_D_by_year.rename(columns={"D": "D_white_hispanic"})
li_D_by_year = li_D_by_year.rename(columns={"D": "D_lowincome_notlowincome"})

county_D_all = (
    bw_D_by_year
    .merge(wh_D_by_year, on="year", how="outer")
    .merge(li_D_by_year, on="year", how="outer")
    .sort_values("year")
    .reset_index(drop=True)
)


######PART 6: QUICK QUALITY ASSURANCE CHECK######

qa_check = (
    school_year_segregation_all_long
    .groupby(["year", "pair"], as_index=False)
    .agg(seg_sum=("seg_contrib", "sum"),
         D=("D", "first"))
)
qa_check["diff"] = qa_check["seg_sum"] - qa_check["D"]

qa_check.head()

,year,pair,seg_sum,D,diff
0,2016,black_white,0.758259,0.758259,0.0
1,2016,lowincome_notlowincome,0.575559,0.575559,0.0
2,2016,white_hispanic,0.602695,0.602695,0.0
3,2017,black_white,0.786513,0.786513,0.0
4,2017,lowincome_notlowincome,0.608481,0.608481,0.0


In [ ]:
######WIDEN THE DATA SO EACH PAIR GETS ITS OWN D COLUMN######

#Pivot so each (school, year, pair) row becomes one row per (school, year),
#with separate columns for the county-year dissimilarity index for each pair
school_year_segregation_all_wide = (
    school_year_segregation_all
    .pivot_table(
        index=[
            "school_id",
            "school_name",
            "year",
            "x_enrollment",
            "group_a",
            "group_b",
            "A_total",
            "B_total",
            "share_A_in_school",
            "share_B_in_school",
            "seg_contrib"
        ],
        columns="pair",
        values="D",
        aggfunc="first" 
    )
    .reset_index()
)

#Clean up the column axis name created by pivot
school_year_segregation_all_wide.columns.name = None


######RENAME D COLUMNS TO NAMES LAY PEOPLE######

school_year_segregation_all_wide = school_year_segregation_all_wide.rename(columns={
    "black_white": "D_black_white",
    "white_hispanic": "D_white_hispanic",
    "lowincome_notlowincome": "D_lowincome_notlowincome"
})

school_year_segregation_all_wide.head()

,school_id,school_name,year,x_enrollment,group_a,group_b,A_total,B_total,share_A_in_school,share_B_in_school,seg_contrib,D_black_white,D_lowincome_notlowincome,D_white_hispanic
0,05-016-2020-17-0001,Evanston Twp High School,2018,3459.0,n_black,n_white,57263,56658.0,0.016677,0.02771,0.005516,0.786462,<NA>,<NA>
1,05-016-2020-17-0001,Evanston Twp High School,2018,3459.0,n_low_income,n_not_low_income,126859,87907.0,0.010468,0.024242,0.006887,<NA>,0.58779,<NA>
2,05-016-2020-17-0001,Evanston Twp High School,2018,3459.0,n_white,n_hispanic,56658,82313.0,0.02771,0.007727,0.009992,<NA>,<NA>,0.623027
3,05-016-2020-17-0001,Evanston Twp High School,2019,3514.0,n_black,n_white,55511,55637.0,0.017276,0.028866,0.005795,0.783907,<NA>,<NA>
4,05-016-2020-17-0001,Evanston Twp High School,2019,3514.0,n_low_income,n_not_low_income,121341,92053.0,0.010574,0.024236,0.006831,<NA>,0.554257,<NA>


In [ ]:
######CREATE A CLEAN WIDE SEGREGATION DATASET (ONE ROW PER SCHOOL-YEAR) ######

school_segregation_all_final = (
    school_year_segregation_all
    .pivot_table(
        index=["school_name", "school_id", "year"],   #only identifiers
        columns="pair",
        values="D",
        aggfunc="first"
    )
    .reset_index()
)

#Remove the pivot column name
school_segregation_all_final.columns.name = None


######PART 1: RENAME THE SEGREGATION COLUMNS ######
school_segregation_all_final = school_segregation_all_final.rename(columns={
    "school_name": "school",
    "black_white": "D_black_white",
    "white_hispanic": "D_white_hispanic",
    "lowincome_notlowincome": "D_lowincome_notlowincome"
})

,school,school_id,year,D_black_white,D_lowincome_notlowincome,D_white_hispanic
0,A B Shepard High Sch (Campus),07-016-2180-16-0007,2018,0.786462,0.58779,0.623027
1,A B Shepard High Sch (Campus),07-016-2180-16-0007,2019,0.783907,0.554257,0.62477
2,A B Shepard High Sch (Campus),07-016-2180-16-0007,2020,0.787379,0.545889,0.623131
3,A B Shepard High Sch (Campus),07-016-2180-16-0007,2021,0.796707,0.549251,0.614866
4,A B Shepard High Sch (Campus),07-016-2180-16-0007,2022,0.796588,0.528412,0.610184


In [84]:
######EXPORT FINAL SCHOOL SEGREGATION DATASET TO CSV######

#Define the folder where the file should be saved
output_path = "/Users/abrahamsadat/My Drive/CoffeeCoders/data/"

#Write the dataframe to a CSV file
school_segregation_all_final.to_csv(
    output_path + "school_segregation_all_final.csv",
    index=False
)

In [ ]:
######CREATING A CLEAN AND MERGED DATA FRAME (NUMBER AND PERCENT OF STUDENTS FOR EACH DEMOGRAPHIC)

#1 Keep only what we need at the school-year level
school_year_demo = (
    school_year_counts[["school_name", "school_id", "year", "x_enrollment", "n_black", "n_white", "n_hispanic", "n_low_income"]]
    .copy()
)

#2 Keep only what we need at the county-year level
county_year_demo = (
    county_year_counts[["year", "x_enrollment", "n_black", "n_white", "n_hispanic", "n_low_income"]]
    .rename(columns={
        "x_enrollment": "county_enrollment",
        "n_black": "county_n_black",
        "n_white": "county_n_white",
        "n_hispanic": "county_n_hispanic",
        "n_low_income": "county_n_low_income"
    })
    .copy()
)

#3 Merge county totals onto each school-year row
school_year_demo = school_year_demo.merge(county_year_demo, on="year", how="left")

#4 Convert counts -> shares (school shares)
school_year_demo["pct_black_school"] = school_year_demo["n_black"] / school_year_demo["x_enrollment"]
school_year_demo["pct_white_school"] = school_year_demo["n_white"] / school_year_demo["x_enrollment"]
school_year_demo["pct_hispanic_school"] = school_year_demo["n_hispanic"] / school_year_demo["x_enrollment"]
school_year_demo["pct_low_income_school"] = school_year_demo["n_low_income"] / school_year_demo["x_enrollment"]

#5 Convert counts -> shares (county shares)
school_year_demo["pct_black_county"] = school_year_demo["county_n_black"] / school_year_demo["county_enrollment"]
school_year_demo["pct_white_county"] = school_year_demo["county_n_white"] / school_year_demo["county_enrollment"]
school_year_demo["pct_hispanic_county"] = school_year_demo["county_n_hispanic"] / school_year_demo["county_enrollment"]
school_year_demo["pct_low_income_county"] = school_year_demo["county_n_low_income"] / school_year_demo["county_enrollment"]

In [ ]:
######REPRESENTATION RATIOS (SCHOOL VS COUNTY) USING LOCATION QUOTIENT (LQ)######

#Black representation
school_year_demo["rr_black"] = (
    school_year_demo["pct_black_school"] /
    school_year_demo["pct_black_county"]
)

#White representation
school_year_demo["rr_white"] = (
    school_year_demo["pct_white_school"] /
    school_year_demo["pct_white_county"]
)

#Hispanic representation
school_year_demo["rr_hispanic"] = (
    school_year_demo["pct_hispanic_school"] /
    school_year_demo["pct_hispanic_county"]
)

#Low-income representation
school_year_demo["rr_low_income"] = (
    school_year_demo["pct_low_income_school"] /
    school_year_demo["pct_low_income_county"]
)

,school_name,school_id,year,x_enrollment,n_black,n_white,n_hispanic,n_low_income,county_enrollment,county_n_black,...,pct_hispanic_school,pct_low_income_school,pct_black_county,pct_white_county,pct_hispanic_county,pct_low_income_county,rr_black,rr_white,rr_hispanic,rr_low_income
0,Evanston Twp High School,05-016-2020-17-0001,2016,3322.0,1010,1438,581,1325,202153.0,53396,...,0.174895,0.398856,0.264137,0.298111,0.356542,0.585364,1.151047,1.45205,0.490531,0.681382
1,Evanston Twp High School,05-016-2020-17-0001,2017,3285.0,966,1449,591,1330,214388.0,58760,...,0.179909,0.404871,0.274083,0.270845,0.373085,0.593802,1.072903,1.628589,0.482219,0.681828
2,Evanston Twp High School,05-016-2020-17-0001,2018,3459.0,955,1570,636,1328,214766.0,57263,...,0.183868,0.383926,0.26663,0.263813,0.383268,0.590685,1.035486,1.720495,0.479737,0.649968
3,Evanston Twp High School,05-016-2020-17-0001,2019,3514.0,959,1606,647,1283,213394.0,55511,...,0.184121,0.365111,0.260134,0.260724,0.387598,0.568624,1.049108,1.752921,0.47503,0.642095
4,Evanston Twp High School,05-016-2020-17-0001,2020,3597.0,935,1647,673,1248,213127.0,54361,...,0.1871,0.346956,0.255064,0.265504,0.391321,0.570899,1.019113,1.724577,0.478125,0.607736


In [ ]:
df = school_year_demo.copy()

#total students by year
year_totals = df.groupby("year")[["n_black","n_white","n_hispanic","x_enrollment"]].sum().reset_index()

#create empty list to store exposure results
exposure_list = []

for yr in df["year"].unique():
    
    sub = df[df["year"] == yr]
    
    X_black = sub["n_black"].sum()
    X_white = sub["n_white"].sum()
    X_hisp  = sub["n_hispanic"].sum()
    
    #school shares
    pct_black = sub["n_black"] / sub["x_enrollment"]
    pct_white = sub["n_white"] / sub["x_enrollment"]
    pct_hisp  = sub["n_hispanic"] / sub["x_enrollment"]
    
    #exposure calculations
    black_white = ((sub["n_black"]/X_black) * pct_white).sum()
    white_black = ((sub["n_white"]/X_white) * pct_black).sum()
    black_hisp  = ((sub["n_black"]/X_black) * pct_hisp).sum()
    hisp_white  = ((sub["n_hispanic"]/X_hisp) * pct_white).sum()
    
    exposure_list.append({
        "year": yr,
        "exp_black_white": black_white,
        "exp_white_black": white_black,
        "exp_black_hisp": black_hisp,
        "exp_hisp_white": hisp_white
    })

exposure_df = pd.DataFrame(exposure_list)

#merge back into original dataframe
school_year_demo = school_year_demo.merge(exposure_df, on="year", how="left")

,school_name,school_id,year,x_enrollment,n_black,n_white,n_hispanic,n_low_income,county_enrollment,county_n_black,...,rr_hispanic,rr_low_income,exp_black_white_x,exp_white_black_x,exp_black_hisp_x,exp_hisp_white_x,exp_black_white_y,exp_white_black_y,exp_black_hisp_y,exp_hisp_white_y
0,Evanston Twp High School,05-016-2020-17-0001,2016,3322.0,1010,1438,581,1325,202153.0,53396,...,0.490531,0.681382,0.106617,0.094466,0.206965,0.194060,0.106617,0.094466,0.206965,0.194060
1,Evanston Twp High School,05-016-2020-17-0001,2017,3285.0,966,1449,591,1330,214388.0,58760,...,0.482219,0.681828,0.088539,0.089597,0.206684,0.172336,0.088539,0.089597,0.206684,0.172336
2,Evanston Twp High School,05-016-2020-17-0001,2018,3459.0,955,1570,636,1328,214766.0,57263,...,0.479737,0.649968,0.087245,0.088176,0.212316,0.169566,0.087245,0.088176,0.212316,0.169566
3,Evanston Twp High School,05-016-2020-17-0001,2019,3514.0,959,1606,647,1283,213394.0,55511,...,0.47503,0.642095,0.087754,0.087555,0.220689,0.168809,0.087754,0.087555,0.220689,0.168809
4,Evanston Twp High School,05-016-2020-17-0001,2020,3597.0,935,1647,673,1248,213127.0,54361,...,0.478125,0.607736,0.087311,0.083878,0.222901,0.171762,0.087311,0.083878,0.222901,0.171762


In [ ]:
#Save dataframe with the location quotient and exposure index to a csv
school_year_demo.to_csv(
    "/Users/abrahamsadat/My Drive/CoffeeCoders/data/segregation_and_exposure.csv",
    index=False
)